In [ ]:
# main.py
from src.data_processor_v2 import DataProcessor
from src.model_trainer import ModelTrainer
from src.trajectory_predictor import TrajectoryPredictor

import numpy as np
import os
import random
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

# --------------------------------------------------
# 1. looking용 config (단일 파일 처리)
# --------------------------------------------------
looking_config = {
    "looking_left01.csv": {"skiprows": 300, "flag": False, "zone": 52, "expected_total_deg": 671.600},
    "looking_left02.csv": {"skiprows": 150, "flag": False, "zone": 52, "expected_total_deg": 1710},
    "looking_left03.csv": {"skiprows": 250, "flag": False, "zone": 52, "expected_total_deg": 2430},
    "looking_left04.csv": {"skiprows": 150, "flag": False, "zone": 52, "expected_total_deg": 2430},
    "looking_left05.csv": {"skiprows": 300, "flag": True,  "zone": 52, "expected_total_deg": 4950},

    "looking_right01.csv": {"skiprows": 200, "flag": False, "zone": 52, "expected_total_deg": -1070},
    "looking_right02.csv": {"skiprows": 100, "flag": False, "zone": 52, "expected_total_deg": -2430},
    "looking_right03.csv": {"skiprows": 200, "flag": False, "zone": 52, "expected_total_deg": -2790},
    "looking_right04.csv": {"skiprows": 350, "flag": True,  "zone": 52, "expected_total_deg": -5220},
}

looking_default_config = {"skiprows": 350, "flag": True, "zone": 52}

pair_config = {
    "looking_left01.csv": {"skiprows": 300, "flag": False, "zone": 52, "expected_total_deg": 671.600},
    "looking_left02.csv": {"skiprows": 150, "flag": False, "zone": 52, "expected_total_deg": 1710},
    "looking_left03.csv": {"skiprows": 250, "flag": False, "zone": 52, "expected_total_deg": 2430},
    "looking_left04.csv": {"skiprows": 150, "flag": False, "zone": 52, "expected_total_deg": 2430},
    "looking_left05.csv": {"skiprows": 300, "flag": True, "zone": 52, "expected_total_deg": 4950},
    
    "looking_right01.csv": {"skiprows": 200, "flag": False, "zone": 52, "expected_total_deg": -1070},
    "looking_right02.csv": {"skiprows": 100, "flag": False, "zone": 52, "expected_total_deg": -2430},
    "looking_right03.csv": {"skiprows": 200, "flag": False, "zone": 52, "expected_total_deg": -2790},
    "looking_right04.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": -5220},
    
    "swing_left01.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": 1980, "heading_corr": True},
    "swing_left02.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": 1980, "heading_corr": True},
    "swing_left03.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": 1890, "heading_corr": True},
    "swing_left04.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": 1980, "heading_corr": True},
    "swing_left05.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": 1890, "heading_corr": True},
    
    "swing_right01.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": -1890, "heading_corr": True},
    "swing_right02.csv": {"skiprows": 150, "skipfooter": 300, "flag": True, "zone": 52, "expected_total_deg": -1980, "heading_corr": True},
    "swing_right03.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": -1980, "heading_corr": True},
    "swing_right04.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": -1890, "heading_corr": True},
    "swing_right05.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": -1890, "heading_corr": True},
    
    "calling_left01.csv": {"skiprows": 350, "skipfooter": 200, "flag": True, "zone": 52, "expected_total_deg": 1890},
    "calling_left02.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": 1980},
    "calling_left03.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": 1980},
    "calling_left04.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": 1890},
    "calling_left05.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": 1980},
    
    "calling_right01.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": -1890},
    "calling_right02.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": -1980},
    "calling_right03.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": -1980},
    "calling_right04.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": -1980},
    "calling_right05.csv": {"skiprows": 350, "flag": True, "zone": 52, "expected_total_deg": -1890},
}

pair_default_config = {"skiprows": 350, "flag": True, "zone": 52}


def load_pair_dataset(sensor_paths, ref_paths, window_size):
    X_list, Y_list, df_list = [], [], []

    if len(sensor_paths) != len(ref_paths):
        raise ValueError("sensor_paths와 ref_paths 길이가 다릅니다.")

    for sensor_path, ref_path in zip(sensor_paths, ref_paths):
        if not os.path.exists(sensor_path):
            print(f"[PAIR] sensor 파일 없음: {sensor_path}")
            continue
        if not os.path.exists(ref_path):
            print(f"[PAIR] ref 파일 없음: {ref_path}")
            continue

        sensor_fname = Path(sensor_path).name
        ref_fname = Path(ref_path).name
        opts = pair_config.get(sensor_fname, pair_default_config)

        try:
            df_all, x, y = DataProcessor.load_and_preprocess_csv_v2(
                file_path_sensor=sensor_path,
                file_path_ref=ref_path,
                **opts,
                window_size=window_size,
            )
            print(f"[PAIR] 완료: {sensor_fname} + {ref_fname}, X={x.shape}, Y={y.shape}")

            df_list.append(df_all)
            X_list.append(x)
            Y_list.append(y)

        except Exception as e:
            print(f"[PAIR] 처리 실패: {sensor_fname} + {ref_fname} -> {e}")

    return df_list, X_list, Y_list

def filter_labels(X, Y, max_deg=100, max_speed=7.0):
    degrees_heading = np.degrees(Y[:, 1])
    mask_heading = np.abs(degrees_heading) <= max_deg
    mask_speed = (Y[:, 0] <= max_speed) & (Y[:, 0] >= 4.5)
    mask = mask_heading & mask_speed

    removed = np.sum(~mask)
    print(f"제거된 샘플 개수: {removed}")

    return X[mask], Y[mask]


def load_looking_dataset(paths, window_size):
    X_list, Y_list, df_list = [], [], []

    for path in paths:
        if not os.path.exists(path):
            print(f"[LOOKING] 파일을 찾을 수 없습니다: {path}")
            continue

        fname = Path(path).name
        opts = looking_config.get(fname, looking_default_config)

        try:
            df_all, x, y = DataProcessor.load_and_preprocess_csv(
                path,
                **opts,
                window_size=window_size,
            )
            print(f"[LOOKING] 완료: {fname}, X={x.shape}, Y={y.shape}")

            df_list.append(df_all)
            X_list.append(x)
            Y_list.append(y)

        except Exception as e:
            print(f"[LOOKING] 처리 실패: {fname} -> {e}")

    return df_list, X_list, Y_list

In [ ]:
BASE_DIR = os.getcwd()
WINDOW_SIZE = 200

SEED = 260223
#SEED = 260330
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def main():
    # ============================================================
    # 1. looking 학습 데이터
    # ============================================================
    learn_looking_paths = [
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left01.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left02.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left03.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left04.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left05.csv"),
        
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right01.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right02.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right03.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right04.csv"),
    ]
    
    learn_pair_sensor_paths = [
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left01.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left02.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left03.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left04.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left05.csv"),
        
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right01.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right02.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right03.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right04.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right05.csv"),
        
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left01.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left02.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left03.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left04.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left05.csv"),
        
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right01.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right02.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right03.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right04.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right05.csv"),
    ]

    learn_pair_ref_paths = [
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left01_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left02_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left03_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left04_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left05_ref.csv"),
        
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right01_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right02_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right03_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right04_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right05_ref.csv"),
        
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left01_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left02_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left03_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left04_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left05_ref.csv"),
    
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right01_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right02_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right03_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right04_ref.csv"),
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right05_ref.csv"),
    ]

    # ============================================================
    # 3. 데이터 로드
    # ============================================================
    df_list = []
    X_list = []
    Y_list = []

    # 3-1. looking
    df_l, X_l, Y_l = load_looking_dataset(learn_looking_paths, WINDOW_SIZE)
    df_list.extend(df_l)
    X_list.extend(X_l)
    Y_list.extend(Y_l)
    
    # 3-2. swing / calling
    df_p, X_p, Y_p = load_pair_dataset(
        learn_pair_sensor_paths,
        learn_pair_ref_paths,
        WINDOW_SIZE,
    )
    df_list.extend(df_p)
    X_list.extend(X_p)
    Y_list.extend(Y_p)


    if len(X_list) == 0 or len(Y_list) == 0:
        raise ValueError("유효한 학습 데이터가 없습니다.")
    
    # ============================================================
    # 4. concatenate
    # ============================================================
    X = np.concatenate(X_list, axis=0)
    Y = np.concatenate(Y_list, axis=0)

    print(f"전체 결합 후 X shape: {X.shape}")
    print(f"전체 결합 후 Y shape: {Y.shape}")
    

    # ============================================================
    # 5. filtering
    # ============================================================
    X_train, Y_train = filter_labels(X, Y, max_deg=100, max_speed=7.5)

    # ============================================================
    # 6. 모델 학습
    # ============================================================
    total_samples, window_size, num_features = X_train.shape
    print(f"총 샘플 수: {total_samples}, 윈도우 크기: {window_size}, 피처 수: {num_features}")

    trainer = ModelTrainer(window_size, num_features, epochs=30, batch_size=256)
    history = trainer.train_model(X_train, Y_train)
    trainer.plot_training_history(history)

    model_path = trainer.save_model()
    
    print("모델이 저장되었습니다:", model_path)


if __name__ == "__main__":
    main()